In [ ]:
import subprocess, sys

packages = [
    "bitsandbytes>=0.42.0",
    "accelerate>=0.27.0",
    "transformers>=4.40.0",
    "huggingface_hub>=0.21.0",
    "sentence-transformers>=2.7.0",
    "langdetect",
]

for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

In [ ]:
import pandas as pd

df = pd.read_csv("/kaggle/input/datasets/raianrashid/master-dataset-1k/Master Dataset 1k.csv")

print(f"Total rows     : {len(df)}")
print(f"Total columns  : {len(df.columns)}")
print()

required_columns = [
    "Word_id",
    "Parts of Speech",
    "English_word",
    "Bengali_source_word",
    "Bangla_script_target",
    "Romanized_target"
]

missing = [col for col in required_columns if col not in df.columns]
if missing:
    print(f"MISSING COLUMNS: {missing}")
else:
    print("All required columns found.")
print()

print("Empty values per column:")
print(df.isnull().sum())
print()

print("Parts of Speech distribution:")
print(df["Parts of Speech"].value_counts())
print()

pd.set_option("display.max_colwidth", 40)
print("First 5 rows:")
print(df.head())

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sentence_transformers import SentenceTransformer, util
from kaggle_secrets import UserSecretsClient

# huggingface
secrets  = UserSecretsClient()
hf_token = secrets.get_secret("HF_TOKEN")
from huggingface_hub import login
login(token=hf_token)

# labse
embedder = SentenceTransformer(
    "sentence-transformers/LaBSE",
    device="cpu"
)
print(f"LaBSE loaded. Dimension: {embedder.get_sentence_embedding_dimension()}")

# 4bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)


MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
# tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=hf_token)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    token=hf_token
)
model.eval()
print("model loaded successfully.")

for i in range(torch.cuda.device_count()):
    allocated = torch.cuda.memory_allocated(i) / 1024**3
    total     = torch.cuda.get_device_properties(i).total_memory / 1024**3
    print(f"GPU {i}: {allocated:.2f} GB used / {total:.2f} GB total")


print("model architecture check")
print(f"  Total layers : {len(model.model.layers)}")
print(f"  Norm type    : {type(model.model.norm).__name__}")

# sanity test
messages = [
    {
        "role": "user",
        "content": (
            "Translate the following word into Bangla. "
            "Respond with only the translated word.\n"
            "Word: book\nTranslation:"
        )
    }
]
prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)
inputs    = tokenizer(prompt, return_tensors="pt").to(model.device)
input_len = inputs["input_ids"].shape[1]

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=20,
        do_sample=False,
        temperature=1.0,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id
    )

decoded = tokenizer.decode(
    output[0][input_len:],
    skip_special_tokens=True
).strip().split("\n")[0]

print(f"Test input  : book")
print(f"Model output: {decoded}")


# labse sanity test
pairs = [
    ("বই", "পুস্তক", "synonym"),
    ("বই", "গাড়ি",  "unrelated"),
    ("গরম", "ঠান্ডা", "antonym"),
]
for w1, w2, label in pairs:
    e1  = embedder.encode(w1, convert_to_tensor=True)
    e2  = embedder.encode(w2, convert_to_tensor=True)
    sim = util.cos_sim(e1, e2).item()
    print(f"  {w1} vs {w2:<12} {sim:.4f}  ({label})")

In [ ]:
import torch
from sentence_transformers import util as st_util

# logit lens
total_layers        = len(model.model.layers)
probe_layer_indices = list(range(total_layers - 10, total_layers))
captured_hidden_states = {}

def make_hook(layer_idx):
    def hook_fn(module, input, output):
        captured_hidden_states[layer_idx] = output[0].detach().float()
    return hook_fn

hooks = []
for idx in probe_layer_indices:
    h = model.model.layers[idx].register_forward_hook(make_hook(idx))
    hooks.append(h)

print(f"Total layers  : {total_layers}")
print(f"Probing layers: {probe_layer_indices}")
print(f"Hooks attached: {len(hooks)}")

#decoding hidden state
def decode_hidden_state(hidden_state, position=-1):
    if hidden_state.dim() == 3:
        h = hidden_state[0, position, :]
    elif hidden_state.dim() == 2:
        h = hidden_state[position, :]
    else:
        raise ValueError(f"Unexpected shape: {hidden_state.shape}")

    h = h.to(
        device=model.lm_head.weight.device,
        dtype=model.lm_head.weight.dtype
    )
    h      = model.model.norm(h.unsqueeze(0))
    logits = model.lm_head(h)
    tok_id = torch.argmax(logits, dim=-1).item()
    raw    = tokenizer.convert_ids_to_tokens([tok_id])[0]
    return raw, tok_id

#bangla detection
def is_bengali(raw_token):
    # method 1, direct Unicode U+0980-U+09FF
    try:
        s = tokenizer.convert_tokens_to_string([raw_token])
        if any('\u0980' <= c <= '\u09FF' for c in s):
            return True
    except Exception:
        pass
    # method 2, Bangla UTF-8 first-byte patterns
    if any(p in raw_token for p in ['à¦', 'à§', 'Ġà¦', 'Ġà§']):
        return True
    # method 3, UTF-8 continuation byte range
    clean = raw_token.lstrip('Ġ▁ ')
    if len(clean) == 1 and '\u0122' <= clean <= '\u0161':
        return True
    return False

#prompts
def make_prompt(source_word, condition):
    if condition == "G1_E":
        content = (
            "Translate the following word into Bangla script. "
            "Respond with only the translated word, nothing else.\n"
            f"Word: {source_word}\nTranslation:"
        )
    elif condition == "G1_B":
        content = (
            "Translate the following Bangla word. "
            "Respond with only the Bangla script translation, nothing else.\n"
            f"Word: {source_word}\nTranslation:"
        )
    elif condition == "G2_E":
        content = (
            "Transliterate the following English word into Romanized Bangla "
            "(write Bangla pronunciation using English letters only).\n"
            "Example: the word 'water' becomes 'pani'.\n"
            "Respond with only the romanized word, nothing else.\n"
            f"Word: {source_word}\nRomanized Bangla:"
        )
    elif condition == "G2_B":
        content = (
            "Transliterate the following Bangla word into Romanized Bangla "
            "(write the pronunciation using English letters only).\n"
            "Example: the word 'পানি' becomes 'pani'.\n"
            "Respond with only the romanized word, nothing else.\n"
            f"Word: {source_word}\nRomanized Bangla:"
        )

    messages = [{"role": "user", "content": content}]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

def get_full_output(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=20,
            do_sample=False,
            temperature=1.0,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
    decoded = tokenizer.decode(
        output[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()
    return decoded.split("\n")[0].strip()

THRESHOLDS    = [0.75, 0.80, 0.85]
THRESHOLD_KEYS = {0.75: "sem_075", 0.80: "sem_080", 0.85: "sem_085"}

def exact_match(actual, ground_truth, condition):
    a = actual.strip()
    g = ground_truth.strip()
    if condition in ["G2_E", "G2_B"]:
        return a.lower() == g.lower()
    return a == g

def semantic_match(actual, ground_truth):
    try:
        e1  = embedder.encode(actual,       convert_to_tensor=True)
        e2  = embedder.encode(ground_truth, convert_to_tensor=True)
        sim = st_util.cos_sim(e1, e2).item()
    except Exception:
        sim = 0.0
    results = {THRESHOLD_KEYS[t]: sim >= t for t in THRESHOLDS}
    return sim, results

#checking for off-targets
def check_off_target(actual_output, condition):
    has_bengali = any('\u0980' <= ch <= '\u09FF' for ch in actual_output)
    if condition in ["G1_E", "G1_B"]:
        return not has_bengali
    else:
        return has_bengali


#function testing using one word
test_prompt = make_prompt("book", "G1_E")
inputs      = tokenizer(test_prompt, return_tensors="pt").to(model.device)
captured_hidden_states.clear()

with torch.no_grad():
    _ = model(**inputs)

output           = get_full_output(test_prompt)
em               = exact_match(output, "বই", "G1_E")
sim, sem_results = semantic_match(output, "বই")

print(f"Output      : {repr(output)}")
print(f"Exact match : {em}")
print(f"LaBSE sim   : {sim:.4f}")
print(f"Sem@0.75    : {sem_results['sem_075']}")
print(f"Sem@0.80    : {sem_results['sem_080']}")
print(f"Sem@0.85    : {sem_results['sem_085']}")
print()

bengali_found = False
for layer_idx in probe_layer_indices:
    if layer_idx in captured_hidden_states:
        raw, _ = decode_hidden_state(captured_hidden_states[layer_idx])
        if is_bengali(raw):
            bengali_found = True
            break

print(f"Bengali at any layer: {bengali_found}")

In [ ]:
import pandas as pd
from tqdm import tqdm

MODEL_LABEL = "qwen2.5-7b"
OUTPUT_PATH = f"/kaggle/working/{MODEL_LABEL}_raw_results.csv"

results    = []
conditions = ["G1_E", "G1_B", "G2_E", "G2_B"]

def save_checkpoint(results, path):
    pd.DataFrame(results).to_csv(path, index=False, encoding="utf-8")

print(f"Running {len(df)} words across {len(conditions)} conditions")
print(f"Total inferences : {len(df) * len(conditions)}")
print(f"Saving to        : {OUTPUT_PATH}")
print()

for i, row in tqdm(df.iterrows(), total=len(df), desc="Words"):
    for condition in conditions:
        source_word  = row["English_word"]         if condition in ["G1_E", "G2_E"] \
                       else row["Bengali_source_word"]
        ground_truth = row["Bangla_script_target"]  if condition in ["G1_E", "G1_B"] \
                       else row["Romanized_target"]

        try:
            prompt = make_prompt(source_word, condition)

            # forward pass for logit lens
            inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
            captured_hidden_states.clear()
            with torch.no_grad():
                _ = model(**inputs)

            # generate actual output
            actual_output = get_full_output(prompt)

            # correctness
            em           = exact_match(actual_output, ground_truth, condition)
            sim, sem_res = semantic_match(actual_output, ground_truth)

            # off-target
            off_target = check_off_target(actual_output, condition)

            # Bengali layer detection
            bengali_any   = False
            bengali_count = 0
            first_bengali = None

            for layer_idx in probe_layer_indices:
                if layer_idx in captured_hidden_states:
                    raw, _ = decode_hidden_state(
                        captured_hidden_states[layer_idx]
                    )
                    if is_bengali(raw):
                        bengali_count += 1
                        bengali_any    = True
                        if first_bengali is None:
                            first_bengali = layer_idx

            results.append({
                "word_id"             : row["Word_id"],
                "pos"                 : row["Parts of Speech"],
                "english_word"        : row["English_word"],
                "bengali_word"        : row["Bengali_source_word"],
                "condition"           : condition,
                "source_word"         : source_word,
                "ground_truth"        : ground_truth,
                "actual_output"       : actual_output,
                "exact_match"         : em,
                "labse_score"         : round(sim, 4),
                "sem_075"             : sem_res["sem_075"],
                "sem_080"             : sem_res["sem_080"],
                "sem_085"             : sem_res["sem_085"],
                "is_off_target"       : off_target,
                "bengali_any_layer"   : bengali_any,
                "bengali_layer_count" : bengali_count,
                "first_bengali_layer" : first_bengali,
                "model"               : MODEL_LABEL
            })

        except Exception as e:
            results.append({
                "word_id"             : row["Word_id"],
                "pos"                 : row["Parts of Speech"],
                "english_word"        : row["English_word"],
                "bengali_word"        : row["Bengali_source_word"],
                "condition"           : condition,
                "source_word"         : source_word,
                "ground_truth"        : ground_truth,
                "actual_output"       : f"ERROR: {str(e)}",
                "exact_match"         : False,
                "labse_score"         : 0.0,
                "sem_075"             : False,
                "sem_080"             : False,
                "sem_085"             : False,
                "is_off_target"       : True,
                "bengali_any_layer"   : False,
                "bengali_layer_count" : 0,
                "first_bengali_layer" : None,
                "model"               : MODEL_LABEL
            })

        if (i + 1) % 50 == 0:
            save_checkpoint(results, OUTPUT_PATH)

save_checkpoint(results, OUTPUT_PATH)
raw_df = pd.DataFrame(results)

print(f"Run complete. Total rows: {len(raw_df)}")
print(f"Saved to: {OUTPUT_PATH}")
print()
print("Quick counts (exact match):")
print(raw_df.groupby("condition")["exact_match"].sum().rename("correct_exact"))
print()
print("Quick counts (semantic @ 0.80):")
print(raw_df.groupby("condition")["sem_080"].sum().rename("correct_sem080"))

In [ ]:
import pandas as pd
import numpy as np

raw_df = pd.read_csv("/kaggle/working/qwen2.5-7b_raw_results.csv")
print(f"Loaded {len(raw_df)} rows")

conditions    = ["G1_E", "G1_B", "G2_E", "G2_B"]
match_columns = {
    "exact"   : "exact_match",
    "sem_075" : "sem_075",
    "sem_080" : "sem_080",
    "sem_085" : "sem_085"
}

all_metrics = []

for condition in conditions:
    subset = raw_df[raw_df["condition"] == condition].copy()
    n      = len(subset)

    for match_label, match_col in match_columns.items():

        ga              = subset[match_col].sum() / n
        ia              = subset["bengali_any_layer"].sum() / n
        off_target_rate = subset["is_off_target"].sum() / n

        # tlp
        n_trans = subset[
            (subset["bengali_any_layer"] == True) &
            (subset[match_col] == False)
        ].shape[0]

        n_task = subset[
            (subset["bengali_any_layer"] == False) &
            (subset[match_col] == False)
        ].shape[0]

        total_failures = n_trans + n_task
        tlp = n_trans / total_failures if total_failures > 0 else 0.0

        all_metrics.append({
            "condition"      : condition,
            "match_method"   : match_label,
            "total_words"    : n,
            "correct"        : int(subset[match_col].sum()),
            "GA_pct"         : round(ga * 100, 2),
            "IA_pct"         : round(ia * 100, 2),
            "off_target_pct" : round(off_target_rate * 100, 2),
            "N_trans"        : n_trans,
            "N_task"         : n_task,
            "total_failures" : total_failures,
            "TLP_pct"        : round(tlp * 100, 2)
        })

metrics_df = pd.DataFrame(all_metrics)


for match_label in match_columns.keys():
    sub = metrics_df[metrics_df["match_method"] == match_label]
    print(f"QWEN2.5-7B — {match_label.upper()}")
    print(f"{'Condition':<10} {'GA%':<8} {'IA%':<8} {'Off-tgt%':<10} "
          f"{'N_trans':<10} {'N_task':<10} {'TLP%':<8}")
    for _, row in sub.iterrows():
        print(f"{row['condition']:<10} {row['GA_pct']:<8} {row['IA_pct']:<8} "
              f"{row['off_target_pct']:<10} {row['N_trans']:<10} "
              f"{row['N_task']:<10} {row['TLP_pct']:<8}")

# script gap

print("SCRIPT GAP ANALYSIS")

print(f"{'Method':<12} {'ENG→Script':<14} {'ENG→Roman':<14} "
      f"{'Gap(Eng)':<12} {'BN→Script':<14} {'BN→Roman':<14} {'Gap(BN)':<10}")


for match_label in match_columns.keys():
    sub = metrics_df[metrics_df["match_method"] == match_label]
    ga  = {r["condition"]: r["GA_pct"] for _, r in sub.iterrows()}
    print(f"{match_label:<12} {ga['G1_E']:<14} {ga['G2_E']:<14} "
          f"{round(ga['G1_E']-ga['G2_E'],2):<12} "
          f"{ga['G1_B']:<14} {ga['G2_B']:<14} "
          f"{round(ga['G1_B']-ga['G2_B'],2):<10}")
print()

# pos breakdown sem_080
print("ACCURACY BY POS — Semantic@0.80")

pos_df    = raw_df.groupby(["condition","pos"])["sem_080"].mean().mul(100).round(2)
pos_pivot = pos_df.unstack(level="pos")
print(pos_pivot.to_string())
print()


METRICS_PATH = "/kaggle/working/qwen2.5-7b_metrics.csv"
metrics_df.to_csv(METRICS_PATH, index=False, encoding="utf-8")
print(f"Metrics saved to: {METRICS_PATH}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

raw_df     = pd.read_csv("/kaggle/working/qwen2.5-7b_raw_results.csv")
metrics_df = pd.read_csv("/kaggle/working/qwen2.5-7b_metrics.csv")

SAVE_DIR    = "/kaggle/working/"
MODEL_LABEL = "Qwen2.5-7B"

CONDITIONS = ["G1_E", "G1_B", "G2_E", "G2_B"]
LABELS = {
    "G1_E" : "G1-E\nEnglish→Bangla script",
    "G1_B" : "G1-B\nBangla→Bangla script",
    "G2_E" : "G2-E\nEnglish→Romanized",
    "G2_B" : "G2-B\nBangla→Romanized"
}
COLORS = {
    "G1_E" : "#4C72B0",
    "G1_B" : "#55A868",
    "G2_E" : "#C44E52",
    "G2_B" : "#DD8452"
}
METHOD_COLORS = {
    "exact"   : "#2d2d2d",
    "sem_075" : "#6baed6",
    "sem_080" : "#2171b5",
    "sem_085" : "#084594"
}
METHOD_LABELS = {
    "exact"   : "Exact Match",
    "sem_075" : "Semantic@0.75",
    "sem_080" : "Semantic@0.80",
    "sem_085" : "Semantic@0.85"
}



# ga across all methods & conditions

fig, ax = plt.subplots(figsize=(13, 6))
methods = ["exact", "sem_075", "sem_080", "sem_085"]
x     = np.arange(len(CONDITIONS))
width = 0.2

for i, method in enumerate(methods):
    sub    = metrics_df[metrics_df["match_method"] == method]
    vals   = [sub[sub["condition"] == c]["GA_pct"].values[0] for c in CONDITIONS]
    offset = (i - 1.5) * width
    bars   = ax.bar(x + offset, vals, width,
                    label=METHOD_LABELS[method],
                    color=METHOD_COLORS[method],
                    edgecolor="white", linewidth=0.5)
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, h + 0.5,
                f"{h:.0f}", ha="center", va="bottom", fontsize=7.5)

ax.set_xticks(x)
ax.set_xticklabels([LABELS[c] for c in CONDITIONS], fontsize=10)
ax.set_ylabel("Generation Accuracy (%)", fontsize=11)
ax.set_title(f"{MODEL_LABEL} — Generation Accuracy by Condition and Evaluation Method",
             fontsize=13, fontweight="bold", pad=15)
ax.set_ylim(0, 115)
ax.legend(fontsize=10)
ax.yaxis.grid(True, linestyle="--", alpha=0.4)
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}qwen_fig1_ga_all_methods.png", dpi=150, bbox_inches="tight")
plt.show()

# tlp across all methods
fig, ax = plt.subplots(figsize=(13, 6))

for i, method in enumerate(methods):
    sub    = metrics_df[metrics_df["match_method"] == method]
    vals   = [sub[sub["condition"] == c]["TLP_pct"].values[0] for c in CONDITIONS]
    offset = (i - 1.5) * width
    bars   = ax.bar(x + offset, vals, width,
                    label=METHOD_LABELS[method],
                    color=METHOD_COLORS[method],
                    edgecolor="white", linewidth=0.5)
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, h + 0.5,
                f"{h:.0f}", ha="center", va="bottom", fontsize=7.5)

ax.axhline(y=50, color="gray", linestyle="--", linewidth=1.2,
           label="50% reference line")
ax.set_xticks(x)
ax.set_xticklabels([LABELS[c] for c in CONDITIONS], fontsize=10)
ax.set_ylabel("Translation Loss Proportion — TLP (%)", fontsize=11)
ax.set_title(f"{MODEL_LABEL} — TLP by Condition and Evaluation Method",
             fontsize=13, fontweight="bold", pad=15)
ax.set_ylim(0, 110)
ax.legend(fontsize=10)
ax.yaxis.grid(True, linestyle="--", alpha=0.4)
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}qwen_fig2_tlp_all_methods.png", dpi=150, bbox_inches="tight")
plt.show()


#  N_trans vs N_task at Sem@0.80

fig, ax = plt.subplots(figsize=(10, 6))

sub     = metrics_df[metrics_df["match_method"] == "sem_080"]
n_trans = [sub[sub["condition"] == c]["N_trans"].values[0] for c in CONDITIONS]
n_task  = [sub[sub["condition"] == c]["N_task"].values[0]  for c in CONDITIONS]
x2      = np.arange(len(CONDITIONS))

p1 = ax.bar(x2, n_trans, 0.5,
            label="N_trans — translation failure",
            color="#4C72B0", edgecolor="white")
p2 = ax.bar(x2, n_task, 0.5, bottom=n_trans,
            label="N_task — comprehension failure",
            color="#C44E52", edgecolor="white")

for i in range(len(CONDITIONS)):
    if n_trans[i] > 0:
        ax.text(i, n_trans[i] / 2, str(n_trans[i]),
                ha="center", va="center", fontsize=11,
                color="white", fontweight="bold")
    if n_task[i] > 0:
        ax.text(i, n_trans[i] + n_task[i] / 2, str(n_task[i]),
                ha="center", va="center", fontsize=11,
                color="white", fontweight="bold")

ax.set_xticks(x2)
ax.set_xticklabels([LABELS[c] for c in CONDITIONS], fontsize=10)
ax.set_ylabel("Number of words", fontsize=11)
ax.set_title(f"{MODEL_LABEL} — Failure Type Breakdown (Semantic@0.80)",
             fontsize=13, fontweight="bold", pad=15)
ax.legend(fontsize=10)
ax.yaxis.grid(True, linestyle="--", alpha=0.4)
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}qwen_fig3_failure_breakdown.png", dpi=150, bbox_inches="tight")
plt.show()


# script Gap at Sem@0.80

fig, ax = plt.subplots(figsize=(9, 6))

sub      = metrics_df[metrics_df["match_method"] == "sem_080"]
ga_g1e   = sub[sub["condition"] == "G1_E"]["GA_pct"].values[0]
ga_g1b   = sub[sub["condition"] == "G1_B"]["GA_pct"].values[0]
ga_g2e   = sub[sub["condition"] == "G2_E"]["GA_pct"].values[0]
ga_g2b   = sub[sub["condition"] == "G2_B"]["GA_pct"].values[0]

sources     = ["English Source", "Bangla Source"]
script_vals = [ga_g1e, ga_g1b]
roman_vals  = [ga_g2e, ga_g2b]
x3          = np.arange(len(sources))
w           = 0.35

bars1 = ax.bar(x3 - w/2, script_vals, w,
               label="Bangla script target",
               color="#4C72B0", edgecolor="white")
bars2 = ax.bar(x3 + w/2, roman_vals, w,
               label="Romanized Bangla target",
               color="#DD8452", edgecolor="white")

for bar in bars1:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2, h + 0.8,
            f"{h:.1f}%", ha="center", va="bottom",
            fontsize=12, fontweight="bold")
for bar in bars2:
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2, h + 0.8,
            f"{h:.1f}%", ha="center", va="bottom",
            fontsize=12, fontweight="bold")

for i, (s, r) in enumerate(zip(script_vals, roman_vals)):
    gap = round(s - r, 1)
    ax.annotate(f"Gap: {gap}%",
                xy=(i, max(s, r) + 7),
                ha="center", fontsize=10,
                fontweight="bold", color="#333333")

ax.set_xticks(x3)
ax.set_xticklabels(sources, fontsize=11)
ax.set_ylabel("Generation Accuracy % (Semantic@0.80)", fontsize=11)
ax.set_title(f"{MODEL_LABEL} — Script Gap: Bangla Script vs Romanized Bangla",
             fontsize=13, fontweight="bold", pad=15)
ax.set_ylim(0, 115)
ax.legend(fontsize=10)
ax.yaxis.grid(True, linestyle="--", alpha=0.4)
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}qwen_fig4_script_gap.png", dpi=150, bbox_inches="tight")
plt.show()


# pos Heatmap at Sem@0.80

pos_df    = raw_df.groupby(["condition","pos"])["sem_080"].mean().mul(100).round(2)
pos_pivot = pos_df.unstack(level="pos")
pos_pivot.index = [LABELS[c].replace("\n", " ") for c in pos_pivot.index]

fig, ax = plt.subplots(figsize=(10, 4))
im = ax.imshow(pos_pivot.values, cmap="Blues", aspect="auto", vmin=0, vmax=100)

ax.set_xticks(range(len(pos_pivot.columns)))
ax.set_xticklabels(pos_pivot.columns, fontsize=12)
ax.set_yticks(range(len(pos_pivot.index)))
ax.set_yticklabels(pos_pivot.index, fontsize=10)

for i in range(len(pos_pivot.index)):
    for j in range(len(pos_pivot.columns)):
        val   = pos_pivot.values[i, j]
        color = "white" if val > 55 else "black"
        ax.text(j, i, f"{val:.1f}%", ha="center", va="center",
                fontsize=12, color=color, fontweight="bold")

plt.colorbar(im, ax=ax, label="Accuracy % (Semantic@0.80)")
ax.set_title(f"{MODEL_LABEL} — Accuracy by Part of Speech (Semantic@0.80)",
             fontsize=13, fontweight="bold", pad=15)
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}qwen_fig5_pos_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

# sensitivity analysis goal 1 only

fig, ax = plt.subplots(figsize=(9, 6))

g1_conditions     = ["G1_E", "G1_B"]
threshold_labels  = ["Exact", "Sem@0.75", "Sem@0.80", "Sem@0.85"]
threshold_methods = ["exact", "sem_075", "sem_080", "sem_085"]

for condition in g1_conditions:
    ga_vals = []
    for method in threshold_methods:
        sub = metrics_df[
            (metrics_df["match_method"] == method) &
            (metrics_df["condition"] == condition)
        ]
        ga_vals.append(sub["GA_pct"].values[0])

    ax.plot(threshold_labels, ga_vals,
            marker="o", linewidth=2.5, markersize=8,
            label=LABELS[condition].replace("\n", " "),
            color=COLORS[condition])

    for x_pos, y_pos in enumerate(ga_vals):
        ax.text(x_pos, y_pos + 1.2, f"{y_pos:.1f}%",
                ha="center", fontsize=9, color=COLORS[condition])

ax.set_ylabel("Generation Accuracy (%)", fontsize=11)
ax.set_title(f"{MODEL_LABEL} — Sensitivity Analysis\n"
             f"GA across evaluation thresholds (Goal 1 conditions)",
             fontsize=13, fontweight="bold", pad=15)
ax.set_ylim(0, 115)
ax.legend(fontsize=10)
ax.yaxis.grid(True, linestyle="--", alpha=0.4)
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}qwen_fig6_sensitivity.png", dpi=150, bbox_inches="tight")
plt.show()


# ga, ia, off-target at Sem@0.80 & g1
fig, ax = plt.subplots(figsize=(9, 6))

sub_g1  = metrics_df[
    (metrics_df["match_method"] == "sem_080") &
    (metrics_df["condition"].isin(["G1_E", "G1_B"]))
]
g1_conds = ["G1_E", "G1_B"]
x4       = np.arange(len(g1_conds))
w2       = 0.25

ga_v = [sub_g1[sub_g1["condition"] == c]["GA_pct"].values[0]         for c in g1_conds]
ia_v = [sub_g1[sub_g1["condition"] == c]["IA_pct"].values[0]         for c in g1_conds]
ot_v = [sub_g1[sub_g1["condition"] == c]["off_target_pct"].values[0] for c in g1_conds]

b1 = ax.bar(x4 - w2, ga_v, w2, label="Generation Accuracy (GA%)",
            color="#4C72B0", edgecolor="white")
b2 = ax.bar(x4,      ia_v, w2, label="Intermediate Accuracy (IA%)",
            color="#55A868", edgecolor="white")
b3 = ax.bar(x4 + w2, ot_v, w2, label="Off-target Rate (%)",
            color="#C44E52", edgecolor="white")

for bars in [b1, b2, b3]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, h + 0.8,
                f"{h:.1f}%", ha="center", va="bottom", fontsize=10)

ax.set_xticks(x4)
ax.set_xticklabels([LABELS[c].replace("\n", " ") for c in g1_conds], fontsize=11)
ax.set_ylabel("Percentage (%)", fontsize=11)
ax.set_title(f"{MODEL_LABEL} — Goal 1: GA, IA and Off-target Rate (Semantic@0.80)",
             fontsize=13, fontweight="bold", pad=15)
ax.set_ylim(0, 115)
ax.legend(fontsize=10)
ax.yaxis.grid(True, linestyle="--", alpha=0.4)
ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig(f"{SAVE_DIR}qwen_fig7_ga_ia_offtarget_g1.png", dpi=150, bbox_inches="tight")
plt.show()